# 02 - Traitement batch avec SparkCore (RDD) - Partie 2
Historique d'evenements Pedali.ma (`trips_history.jsonl`, copie dans `s3a://lake/bronze/history/` par `scripts/load_history.sh`).

In [1]:
import sys, time
sys.path.append('/home/jovyan/work/pipeline'); sys.path.append('/home/jovyan/work/data')
from pedali_spark import get_spark
spark = get_spark('pedali-02-rdd')
sc = spark.sparkContext
print('Spark', spark.version, '| master', sc.master, '| defaultParallelism', sc.defaultParallelism)

Spark 3.5.3 | master spark://spark-master:7077 | defaultParallelism 4


## 1. Chargement : `sc.textFile()` + `json.loads`
Transformations **lazy** : rien n'est lu tant qu'une action n'est pas appelee.

In [2]:
import json
HISTORY = 's3a://lake/bronze/history/trips_history.jsonl'
raw = sc.textFile(HISTORY).filter(lambda l: l.strip()).map(json.loads)
print('evenements bruts :', raw.count(), '(action)')
print(raw.first())

evenements bruts : 50000 (action)


{'trip_id': 'T-00003430', 'user_id': 'U-04989', 'vehicle_id': 'V-EB-0877', 'vehicle_type': 'EBIKE', 'start_station': 'S-CAS-001', 'end_station': None, 'city': 'Casablanca', 'status': 'RESERVED', 'duration_s': None, 'distance_km': None, 'battery_pct': 50, 'price_mad': None, 'event_time': '2026-10-17T00:15:40Z'}


## 2. Dedoublonnage puis `filter` : trajets termines (`ENDED`)
Le generateur re-emet ~2 % des evenements (meme `trip_id`, meme `status`) : sans dedoublonnage le chiffre d'affaires serait gonfle. Le `reduceByKey` sur `(trip_id, status)` provoque un **shuffle**.

In [3]:
events = raw.map(lambda e: ((e['trip_id'], e['status']), e)).reduceByKey(lambda a, b: a).values()
ended = events.filter(lambda e: e['status'] == 'ENDED').cache()
print('apres dedoublonnage :', events.count(), '| trajets termines :', ended.count())

apres dedoublonnage : 48997 | trajets termines : 14880


## 3. Chiffre d'affaires par ville : `map` + `reduceByKey`

In [4]:
ca = ended.map(lambda e: (e['city'], e['price_mad'])).reduceByKey(lambda a, b: a + b).collect()
for ville, v in sorted(ca): print(f'{ville:12s} {v:10.2f} MAD')

Casablanca     75430.27 MAD
Marrakech      51959.36 MAD
Rabat          43473.12 MAD
Tanger         42600.26 MAD


## 4. Duree moyenne par ville et type de vehicule
`aggregateByKey` calcule la **somme et le nombre en un seul passage** (pas de second parcours).

In [5]:
avg = (ended.map(lambda e: ((e['city'], e['vehicle_type']), e['duration_s']))
       .aggregateByKey((0, 0), lambda acc, d: (acc[0] + d, acc[1] + 1), lambda x, y: (x[0] + y[0], x[1] + y[1]))
       .mapValues(lambda s: (round(s[0] / s[1] / 60, 2), s[1])).collectAsMap())
print(f"{'ville':12s} {'type':8s} {'duree moy. (min)':>17s} {'trajets':>8s}")
for (ville, t), (m, n) in sorted(avg.items()): print(f'{ville:12s} {t:8s} {m:17.2f} {n:8d}')

ville        type      duree moy. (min)  trajets
Casablanca   BIKE                 18.15     2588
Casablanca   EBIKE                18.27     1609
Casablanca   SCOOTER              18.16     1071
Marrakech    BIKE                 18.23     1829
Marrakech    EBIKE                18.57     1086
Marrakech    SCOOTER              18.32      704
Rabat        BIKE                 18.15     1485
Rabat        EBIKE                18.30      903
Rabat        SCOOTER              18.15      643
Tanger       BIKE                 18.14     1448
Tanger       EBIKE                18.47      935
Tanger       SCOOTER              18.34      579


## 5. Top 5 des stations de depart : `takeOrdered`

In [6]:
top5 = ended.map(lambda e: (e['start_station'], 1)).reduceByKey(lambda a, b: a + b).takeOrdered(5, key=lambda kv: (-kv[1], kv[0]))
for s, n in top5: print(s, n)

S-CAS-005 343
S-CAS-097 93
S-CAS-096 87
S-MAR-007 87
S-CAS-084 85


## 6. Lineage : `toDebugString()`
- **Transformations (lazy)** : `textFile`, `filter`, `map`, `reduceByKey`, `values`, `aggregateByKey`, `mapValues`, `cache`.
- **Actions** : `count`, `first`, `collect`, `collectAsMap`, `takeOrdered`.

Chaque `reduceByKey` / `aggregateByKey` cree une frontiere de **shuffle** (`ShuffledRDD`), donc un nouveau stage ; `filter` et `map` restent dans le meme stage (dependances etroites).

In [7]:
print(ended.toDebugString().decode())

(2) PythonRDD[8] at RDD at PythonRDD.scala:53 [Memory Serialized 1x Replicated]
 |       CachedPartitions: 2; MemorySize: 1102.3 KiB; DiskSize: 0.0 B
 |  MapPartitionsRDD[7] at mapPartitions at PythonRDD.scala:160 [Memory Serialized 1x Replicated]
 |  ShuffledRDD[6] at partitionBy at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 +-(2) PairwiseRDD[5] at reduceByKey at /tmp/ipykernel_14043/3594246375.py:1 [Memory Serialized 1x Replicated]
    |  PythonRDD[4] at reduceByKey at /tmp/ipykernel_14043/3594246375.py:1 [Memory Serialized 1x Replicated]
    |  s3a://lake/bronze/history/trips_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
    |  s3a://lake/bronze/history/trips_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]


## 7. DAG, stages et tolerance aux pannes
Les jobs et stages sont visibles dans la Spark UI (http://localhost:4040 pendant l'execution). Resume des jobs ci-dessous.

**Tolerance aux pannes** : un RDD est immuable et garde son *lineage* (la recette de ses transformations). Si une partition est perdue (executor en panne), Spark la **recalcule** a partir de ses parents sans avoir besoin de repliquer les donnees ; le `cache()` evite de refaire ce calcul a chaque action.

In [8]:
import json as _j, urllib.request as _u
base = f'http://localhost:4040/api/v1/applications/{sc.applicationId}'
for j in sorted(_j.load(_u.urlopen(base + '/jobs')), key=lambda x: x['jobId']):
    print(f"job {j['jobId']:>2} {j['status']:<9} stages={sorted(j['stageIds'])} taches={j['numTasks']:>3}  {j['name'][:55]}")

job  0 SUCCEEDED stages=[0] taches=  2  count at /tmp/ipykernel_14043/2975042247.py:4
job  1 SUCCEEDED stages=[1] taches=  1  runJob at PythonRDD.scala:181
job  2 SUCCEEDED stages=[2, 3] taches=  4  count at /tmp/ipykernel_14043/3594246375.py:3
job  3 SUCCEEDED stages=[4, 5] taches=  4  count at /tmp/ipykernel_14043/3594246375.py:3
job  4 SUCCEEDED stages=[6, 7, 8] taches=  6  collect at /tmp/ipykernel_14043/391903125.py:1
job  5 SUCCEEDED stages=[9, 10, 11] taches=  6  collectAsMap at /tmp/ipykernel_14043/3082896384.py:3
job  6 SUCCEEDED stages=[12, 13, 14] taches=  6  takeOrdered at /tmp/ipykernel_14043/255431529.py:1


In [9]:
spark.stop()